# Group 81 Objective 2.1
Birat Timsina. Eight differences describing competitive form and match context. This executed notebook uses the same project.py functions as the scripts. Read REQUIREMENTS_AND_OVERLAP.md before submission.

In [1]:
import project as p
import pandas as pd
matches, history = p.load_data()
print("Official matches:", len(matches))
display(matches.head())

match_id,match_date,kickoff_utc,team_a,team_b,goals_a,goals_b,shootout_a,shootout_b,stage,goal_difference
400021443,2026-06-11,2026-06-11T19:00:00Z,Mexico,South Africa,2,0,NaN,NaN,First Stage,2
400021441,2026-06-11,2026-06-12T02:00:00Z,South Korea,Czech Republic,2,1,NaN,NaN,First Stage,1
400021449,2026-06-12,2026-06-12T19:00:00Z,Canada,Bosnia and Herzegovina,1,1,NaN,NaN,First Stage,0
400021458,2026-06-12,2026-06-13T01:00:00Z,United States,Paraguay,4,1,NaN,NaN,First Stage,3
400021447,2026-06-13,2026-06-13T19:00:00Z,Qatar,Switzerland,1,1,NaN,NaN,First Stage,0


Official matches: 104


## Build the eight features
All date comparisons use strictly earlier dates. The target is A goals minus B goals, including extra time and excluding shootouts. The model file has one ID, eight X columns and y.

In [2]:
data = p.build_features(matches, history)
print("Dataset shape:", data.shape)
display(data.head())
display(pd.read_csv(p.ROOT / "data/feature_dictionary.csv")[["name", "construction"]])

match_id,competitive_form_gap,clean_sheet_gap,scoring_failure_gap,margin_volatility_gap,form_momentum_gap,schedule_strength_gap,workload_30d_gap,world_cup_experience_gap,goal_difference
400021443,0.8,0.2,-0.1,-0.351099,2.220446e-16,0.52,1.0,2.484907,2
400021441,0.4,-0.1,0.0,-1.411244,4.000000e-01,0.34,0.0,2.397895,1
400021449,-0.1,0.1,0.2,0.434491,1.800000e+00,0.43,0.0,0.000000,0
400021458,0.0,-0.2,-0.2,0.963693,8.000000e-01,0.41,1.0,2.197225,3
400021447,0.0,-0.2,0.1,0.787602,-1.600000e+00,-0.07,0.0,-1.178655,0


name,construction
Competitive form,"Average result points from the previous ten selected competitive internationals. Award 3 for a win, 1 for a draw and 0 for a loss, regardless of competition format."
Clean sheet rate,Fraction of the previous ten competitive matches in which the team conceded zero goals.
Scoring failure rate,Fraction of the previous ten competitive matches in which the team scored zero goals.
Margin volatility,"Sample standard deviation of goals scored minus goals conceded over the previous ten competitive games, using n minus 1 in the variance denominator."
Form momentum,Mean result points in the most recent five competitive games minus the mean in the preceding five. Then subtract the two teams' momentum values.
Past opponent strength,"For each of the ten previous competitive opponents, calculate its points per game in its own ten competitive games BEFORE that historical meeting. Average those ten opponent values."
Recent workload,"Count all recorded internationals in the interval from 30 days before the prediction date up to, but excluding, that date. Include friendlies for this feature."
World Cup experience,"Count matches played in the 2014, 2018 and 2022 World Cups and apply the natural logarithm of one plus that count. Subtract B from A."


Dataset shape: (104, 10)


## Trace a worked example
The component file contains each team's value before subtraction. The main audit lists the ten competitive games. The separate opponent audit lists the ten games used for each historical opponent.

In [3]:
first_id = matches.match_id.iloc[0]
components = pd.read_csv(p.ROOT / "data/team_components_208.csv", dtype={"match_id": str})
audit = pd.read_csv(p.ROOT / "data/rolling_history_2080.csv", dtype={"match_id": str})
display(components[components.match_id == first_id])
display(audit[audit.match_id == first_id][["side", "date", "team", "opponent", "gf", "ga", "points", "opponent_prior_ppg"]])

match_id,prediction_date,team,side,first_history_date,last_history_date,history_n,world_cup_matches_2014_2022,competitive_form_gap,clean_sheet_gap,scoring_failure_gap,margin_volatility_gap,form_momentum_gap,schedule_strength_gap,workload_30d_gap,world_cup_experience_gap
400021443,2026-06-11,Mexico,a,2024-11-15,2025-07-06,10,11,2.5,0.6,0.2,1.549193,0.2,1.87,3,2.484907
400021443,2026-06-11,South Africa,b,2025-03-21,2026-01-04,10,0,1.7,0.4,0.3,1.900292,0.2,1.35,2,0.000000


side,date,team,opponent,gf,ga,points,opponent_prior_ppg
a,2024-11-15,Mexico,Honduras,0,2,0,1.9
a,2024-11-19,Mexico,Honduras,4,0,3,1.9
a,2025-03-20,Mexico,Canada,2,0,3,1.5
a,2025-03-23,Mexico,Panama,2,1,3,1.9
a,2025-06-14,Mexico,Dominican Republic,3,2,3,2.4
a,2025-06-18,Mexico,Suriname,2,0,3,1.4
a,2025-06-22,Mexico,Costa Rica,0,0,1,2.3
a,2025-06-28,Mexico,Saudi Arabia,2,0,3,1.3
a,2025-07-02,Mexico,Honduras,1,0,3,2.2
a,2025-07-06,Mexico,United States,2,1,3,1.9


## Fit the model
Scaling is learned from the training subset only. No intercept and no centering ensure that swapping teams reverses the prediction. The 22 later games were previously inspected in version 1, so this is a retrospective evaluation.

In [4]:
train, later, folds = p.splits(matches)
model = p.model().fit(data.loc[train, p.FEATURES], data.loc[train, p.TARGET])
prediction = model.predict(data.loc[later, p.FEATURES])
print(p.scores(data.loc[later, p.TARGET], prediction))
display(pd.read_csv(p.ROOT / "results/metrics.csv"))

model,split,MAE,RMSE,R2
Eight feature OLS,training,1.262779,1.673807,0.353374
Eight feature OLS,later matches,1.419575,1.591540,0.054764
Always zero,later matches,1.409091,1.637626,-0.000771
Training mean,later matches,1.465632,1.767819,-0.166221


{'MAE': 1.4195750143301729, 'RMSE': 1.5915397414034618, 'R2': 0.054763766956180904}


## Evaluate each feature
Positive removal delta means losing the feature worsens validation MAE. Read both fold deltas because pooled improvements can hide instability.

In [5]:
display(pd.read_csv(p.ROOT / "results/feature_ablation.csv"))
display(pd.read_csv(p.ROOT / "results/sensitivity.csv"))

feature,label,full_cv_MAE,removed_cv_MAE,delta_MAE,fold_1_delta,fold_2_delta
competitive_form_gap,Competitive form,1.2144,1.150748,-0.063652,-0.136721,0.055084
clean_sheet_gap,Clean sheets,1.2144,1.209285,-0.005115,-0.019976,0.019034
scoring_failure_gap,Scoring failure,1.2144,1.243338,0.028938,0.038134,0.013994
margin_volatility_gap,Margin volatility,1.2144,1.186767,-0.027633,-0.050668,0.009798
form_momentum_gap,Form momentum,1.2144,1.137705,-0.076695,-0.126131,0.003640
schedule_strength_gap,Past opponent strength,1.2144,1.203761,-0.010639,-0.000045,-0.027855
workload_30d_gap,30 day workload,1.2144,1.187560,-0.026840,-0.028516,-0.024117
world_cup_experience_gap,World Cup experience,1.2144,1.336738,0.122338,0.367766,-0.276483


variant,cv_MAE
6 competitive matches,1.428894
10 competitive matches,1.214400
14 competitive matches,1.268005
10 internationals including friendlies,1.079161
10 competitive matches with intercept,1.161087


![Feature removal results](charts/ablation.png)

## Inspect prediction errors
The model is slightly worse on MAE and better on RMSE than the zero baseline. Do not report this as high accuracy.

In [6]:
display(pd.read_csv(p.ROOT / "results/sequential_metrics.csv"))

model,MAE,RMSE,R2
Expanding OLS,1.254298,1.574116,0.289219
Always zero,1.484375,1.875000,-0.008474


![Baseline comparison](charts/baselines.png)

![Prediction diagnostics](charts/diagnostics.png)

## Verify integrity
These tests check data shape and temporal logic. They do not certify the correctness of every historical source result or unseen course requirements.

In [7]:
import unittest
import test_project
suite = unittest.defaultTestLoader.loadTestsFromModule(test_project)
check = unittest.TextTestRunner(verbosity=1).run(suite)
assert check.wasSuccessful()

............
----------------------------------------------------------------------
Ran 12 tests in 2.944s

OK
